# Module 3: Deploying Environments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/openenv-course/blob/main/module-3/notebook.ipynb)

In this notebook, you'll:
1. Clone an existing OpenEnv environment
2. Modify it locally
3. Test it with uvicorn
4. Optionally deploy to HF Spaces

## Prerequisites

For local testing:
- Python 3.10+
- Docker (optional)

For deployment:
- Hugging Face account
- HF token with write access

## Setup

In [ ]:
!pip install openenv-core fastapi uvicorn -q

## 1. Clone an Existing Environment

Let's clone the Echo environment as a starting point.

In [ ]:
!git clone https://huggingface.co/spaces/openenv/echo-env
%cd echo-env
!ls -la

## 2. Explore the Structure

Every OpenEnv environment has the same structure.

In [ ]:
!tree -L 2 .

## 3. Modify the Environment

Let's add a new tool to the Echo environment.

In [ ]:
# Read the current environment code
with open('echo_env/server/environment.py', 'r') as f:
    env_code = f.read()
    print(env_code[:500])  # Show first 500 chars

## 4. Test Locally with Uvicorn

Start the server in the background (in a real terminal, you'd run this in a separate window):

In [ ]:
# In a real environment, you would run:
# uvicorn echo_env.server.app:app --host 0.0.0.0 --port 8000 --reload

# For this notebook, we'll test the health endpoint
import requests
import time

# Assuming server is running on localhost:8000
try:
    response = requests.get('http://localhost:8000/health')
    print(f"Health check: {response.json()}")
except Exception as e:
    print(f"Server not running locally. Error: {e}")
    print("\nTo run the server, open a terminal and execute:")
    print("uvicorn echo_env.server.app:app --host 0.0.0.0 --port 8000 --reload")

## 5. Test with the Client

Connect to your local server.

In [ ]:
import sys
sys.path.insert(0, '.')

from echo_env import EchoEnv

# Try connecting to local server
try:
    with EchoEnv(base_url="http://localhost:8000").sync() as env:
        response = env.call_tool("echo_message", message="Testing local deployment!")
        print(f"Response: {response}")
except Exception as e:
    print(f"Could not connect to local server: {e}")
    print("\nFalling back to hosted version...")
    
    # Fallback to hosted version
    with EchoEnv(base_url="https://openenv-echo-env.hf.space").sync() as env:
        response = env.call_tool("echo_message", message="Testing hosted deployment!")
        print(f"Response: {response}")

## 6. Docker Deployment (Optional)

If you have Docker installed, you can build and run the container.

In [ ]:
# Build the Docker image
!docker build -t my-echo-env:latest -f echo_env/server/Dockerfile .

# Run the container
!docker run -d -p 8000:8000 --name echo-env-container my-echo-env:latest

# Check if it's running
!docker ps | grep echo-env

## 7. Deploy to HF Spaces (Optional)

To deploy your modified environment to Hugging Face Spaces:

```bash
# Login to HF
huggingface-cli login

# Push to a new Space
openenv push --repo-id yourusername/my-echo-env
```

Your environment will be available at:
- API: `https://yourusername-my-echo-env.hf.space`
- Docs: `https://yourusername-my-echo-env.hf.space/docs`
- Web UI: `https://yourusername-my-echo-env.hf.space/web`

## Key Takeaways

1. **Three deployment options**: Uvicorn (fast iteration), Docker (isolation), HF Spaces (public access)
2. **Same interface everywhere**: Your client code works with local and hosted deployments
3. **Quick iteration**: `--reload` flag makes development fast
4. **One-command deployment**: `openenv push` handles the entire deployment

In Module 4, you'll build your own environment from scratch using the same pattern.

## Exercise: Customize and Deploy

Try:
1. Add a new tool to the Echo environment
2. Test it locally
3. Deploy your customized version to HF Spaces
4. Share the URL with others to try your environment

In [ ]:
# Your code here
